In [23]:
!pip install pdfplumber
!pip install "transformers<5"


In [24]:
#Asking for user input
path = input("Enter the path of your resume")

# Extract text
import pdfplumber
with pdfplumber.open(f"{path}") as pdf:
  fullText = ""
  for page in pdf.pages:
    text=page.extract_text()
    fullText+=text

#Summarization(Pipeline)
from transformers import pipeline

summarizer = pipeline("summarization",truncation=True)

summaryPipeline = summarizer(fullText,max_length=130)

#Summarization(AutoModel)
from transformers import AutoModelForSeq2SeqLM,AutoTokenizer
checkpoint="facebook/bart-large-cnn"
tokenizer=AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)
inputs = tokenizer(fullText,truncation=True,max_length=1024,return_tensors="pt")
outputs = model.generate(**inputs)
summary = tokenizer.decode(outputs[0],skip_special_tokens=True)

#Tone Analysis
tone = pipeline("zero-shot-classification",truncation=True)
resumeTone=tone(fullText, candidate_labels=["confident", "formal", "humble", "enthusiastic", "casual"])

#Skill Classification
skills = pipeline("zero-shot-classification",truncation=True,candidate_labels=["AI","Python","Software Development","Communication", "Leadership", "Project Management"],multi_label=True)
classifiedSkills = skills(fullText)

print("Summary of the resume is(from the pipeline)",summaryPipeline)
print("Summary of the resume is(from the AutoModel)",summary)
print("Tone of the resume is",resumeTone['labels'],resumeTone['scores'])
print("Skills are",classifiedSkills['labels'],classifiedSkills['scores'])


Enter the path of your resumeResume.pdf


No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu
No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1 (https://huggingface.co/facebook/bart-large-mnli).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu
No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1 (https://huggingface.co/facebook/bart-large-mnli).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


Summary of the resume is(from the pipeline) [{'summary_text': ' Rishidpatel: "Dedicated and results-driven professional with a proven track record of hard work and passion" "I am deeply committed to continuous learning and adapting to new challenges" "Recognized for my strong work ethic and ability to excel in a fast-paced environment, I am adept at multitasking and multitasking seamlessly within a team"'}]
Summary of the resume is(from the AutoModel) Rishid Patel is a dedicated and results-driven professional with a proven track record of hard work and passion. I am deeply committed to continuous learning and adapting to new challenges. He is adept at multitasking and seamlesslycollaborating seamlessly within a team. He has developed an interactive Simon game application using HTML, CSS and JavaScript.
Tone of the resume is ['formal', 'enthusiastic', 'confident', 'humble', 'casual'] [0.27673470973968506, 0.25951921939849854, 0.2369922250509262, 0.12973925471305847, 0.09701453894376755